### This is code for the exercise in the book of MLP which is used to recognize the hand-writting numbers

#### Difficulty(1): as the layer change to 2, so the code should match the number of layer.
#### Solution(1): delete the loop in evaluate function.

#### Finding: the learning efficeincy of network with jsut two layers is not as good as network with three layers.

In [1]:
import random
import numpy as np

In [2]:
def sigmoid(z):
    """sigmoid function."""
    return 1.0/(1.0+np.exp(-z))

In [3]:
def sigmoid_prime(z):
    """Derivative the sigmoid functions."""
    return sigmoid(z) * (1 - sigmoid(z))

In [4]:
class Network(object):
    def __init__(self, sizes):
        """initialize the bias and weights"""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes [1:]]
        self.weights = [np.random.randn(y, x) for x, y in zip(sizes[:-1], sizes[1:])]

    def feedforward(self, a):
        """Return the output of the networks if 'a' is a input"""
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a) + b)
        return a


    def SGD(self, training_data, epochs, mini_batch_size, eta, test_data=None):
        """Train the neural network using mini-batch stochastic gradient descent"""
        if test_data:
            n_test = len(test_data)
            n = len(training_data)
        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k+mini_batch_size] for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
            if test_data:
                print ("Epoch {0}: {1}/{2}".format(j, self.evaluate(test_data), n_test))
            else:
                print ("Epoch {0} complete".format(j))


    def update_mini_batch(self, mini_batch, eta):
        """update the network's weights and biases by applying gradient descent using backpropagation to a single minibatch."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
        self.weights = [w - (eta/len(mini_batch))*nw for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta/len(mini_batch))*nb for b, nb in zip(self.biases, nabla_b)]

         
    def backprop(self, x, y):
        """return a tuple''(nabla_b, nabla_w)''representing the gradient for the cost function C_x.''nabla_b''and ''nabla_w''are 
        layer-by-layer lists  of numpy arrays, similar to''self.biases''and ''self.weights''."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
    
        # feedforward
        activation = x
        activations = [x] # list to store all the activations, layer by layer
        zs = [] # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        
        # backward pass
        delta = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())
        
        return nabla_b, nabla_w
   
    
    def evaluate(self, test_data):
        """Return the number of the test inouts for which the neural network outputs the correct results"""
        test_results = []
        for (x, y) in test_data:
            # y is a one-hot vector so armax will be recovered to a real number
            if isinstance(y, np.ndarray):
                if y.size == 10:
                    true_label = int(np.argmax(y))
                else:
                    true_label = int(y.flatten()[0])
            else:
                true_label = int(y)

            # armax can choose the maximum of x
            prediction = int(np.argmax(self.feedforward(x)))
            test_results.append((prediction, true_label))
    
        # 使用 p (prediction) 和 l (label) 避免与外层 x, y 混淆
        return sum(int(p == l) for (p, l) in test_results)

    def cost_derivative(self, output_activations, y):
        """ Return the vector of partial derivative of a about C_x"""
        return (output_activations - y)

In [5]:
"""cause i met some difficulties to loading the data ftom the website which 
author provided so i input training data and test data from MNIST directly."""
# input data from MNIST
import numpy as np
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1, cache=True, parser='auto')

# Normalize pixel value to [0,1]
X = mnist.data.values / 255.0
# convert label from string to integer
y = mnist.target.values.astype(int)

# broke images and labels to training data and test data
X_train, X_test = X[:50000], X[50000:]
y_train, y_test = y[:50000], y[50000:]

# change x from 1D array to vector, y from integer to one-hot encoding
def format_data(X, y):
    result = []
    for i in range(len(X)):
        img_vector = X[i].reshape(-1, 1)
        
      
        label_vector = np.zeros((10, 1))
        # resure it is a integer
        label_num = int(y[i])
        label_vector[label_num] = 1.0
        result.append((img_vector, label_vector))
    return result
    
training_data = format_data(X_train, y_train)
test_data = format_data(X_test, y_test)

print("Training data size:", len(training_data))
print("Test data size:", len(test_data))

Training data size: 50000
Test data size: 20000


In [6]:
net = Network([784, 10])
net.SGD(training_data, 5, 10, 1.0, test_data = test_data)

Epoch 0: 9609/20000
Epoch 1: 13049/20000
Epoch 2: 13155/20000
Epoch 3: 13390/20000
Epoch 4: 14989/20000
